Test on the Orders all sources

In [0]:
df = spark.table("hub_prd.g_isc.v_sales_orders_all_sources")
size = df.count()
size
display(df)

In [0]:
df_orders_filtered1 = df.filter(df.MaterialKeyReltio.contains("MMS"))
df_orders_filtered2 = df.filter(df.MaterialKeyReltio.contains("MDS"))

count_filtered1 = df_orders_filtered1.count()
count_filtered2 = df_orders_filtered2.count()

df_appended = df_orders_filtered1.union(df_orders_filtered2)
count_appended = df_appended.count()

if count_appended == (count_filtered1 + count_filtered2):
    print("ok")

accessing reltioo and ipd products


In [0]:
df_hier = spark.read.format("delta").load("/Volumes/hub_sbx/g_ipd/classifications/g_reltioproduct_hier/")
filtered_df_hier = df_hier.filter(
    ((df_hier.BusinessUnit == "MMS") & (df_hier.FinSubBusinessUnitDescription == "Infusion Systems")) |
    ((df_hier.BusinessUnit == "MDS") & (df_hier.FinSubBusinessUnitDescription == "Infusion Preparation & Delivery"))
)
display(filtered_df_hier)

In [0]:
from pyspark.sql.functions import concat_ws

filtered_df_hier = filtered_df_hier.withColumn("BusinessUnit_SKU", concat_ws("|", filtered_df_hier.BusinessUnit, filtered_df_hier.SKU))
display(filtered_df_hier)

checking matching sku to the orders

In [0]:
unique_ipd_sku = filtered_df_hier.select("BusinessUnit_SKU").distinct()
material_key_reltio = df_appended.select("MaterialKeyReltio").distinct()

display(unique_ipd_sku)
display(material_key_reltio)

In [0]:
# Count of lines in both DataFrames
count_unique_ipd_sku = unique_ipd_sku.count()
count_material_key_reltio = material_key_reltio.count()

# Join the two DataFrames on the matching key
matched_df = unique_ipd_sku.join(material_key_reltio, unique_ipd_sku.BusinessUnit_SKU == material_key_reltio.MaterialKeyReltio, "inner")

# Count of matched lines
count_matched = matched_df.count()

# Count of unmatched lines in unique_ipd_sku
count_unmatched_ipd_sku = count_unique_ipd_sku - count_matched

# Count of unmatched lines in material_key_reltio
count_unmatched_material_key_reltio = count_material_key_reltio - count_matched

# Display results
display(unique_ipd_sku)
display(material_key_reltio)
display(matched_df)

print(f"Count unique_ipd_sku: {count_unique_ipd_sku}")
print(f"Count material_key_reltio: {count_material_key_reltio}")
print(f"Count matched: {count_matched}")
print(f"Count unmatched in unique_ipd_sku: {count_unmatched_ipd_sku}")
print(f"Count unmatched in material_key_reltio: {count_unmatched_material_key_reltio}")

ùnmatched products from reltio - need to check why - if they have no rev o or they are outside of IPD - like dispednsin - or non material products - like servikce etc

In [0]:
# Find unmatched material_key_reltio
unmatched_material_key_reltio = material_key_reltio.join(unique_ipd_sku, material_key_reltio.MaterialKeyReltio == unique_ipd_sku.BusinessUnit_SKU, "left_anti")

# Display unmatched material_key_reltio
display(unmatched_material_key_reltio)

filtwering ipd products out ofd the sales orders all sources


changing ordertype to get lots codes

In [0]:
from pyspark.sql.functions import when, col, substring

df_2 = df.withColumn(
    "OrderType2",
    when(col("SourceSystem") == "LOT", substring(col("OrderNumber"), 1, 3))
    .otherwise(col("OrderType"))
)

display(df_2)

showing the orders codes per order quantity within IPD filtered ordered to put them into classification in the excel file

Test on the inventory

filter IPD